# Run MoneyPrinterTurbo on Google Colab

This notebook installs the Equilibriumpress fork in an isolated Python 3.11 environment. It can launch the Streamlit WebUI or a GitHub Issues video queue. Colab runtimes are temporary, so generated files disappear when the runtime resets unless you download them.

## 1. Install MoneyPrinterTurbo

The setup is safe to run again. It clones the fork on the first run and updates it on later runs.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/Equilibriumpress/MoneyPrinterTurbo.git"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True
    )
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True
    )

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "uv", "pyngrok"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
subprocess.run(["uv", "sync", "--frozen", "--python", "3.11"], check=True)
print(f"MoneyPrinterTurbo is ready in {REPO_DIR}")

## 2. Configure ngrok

Create an ngrok account and copy its authentication token. The next cell reads the token without displaying or storing it.

In [ ]:
from getpass import getpass

from pyngrok import ngrok

ngrok.kill()
ngrok_token = getpass("Enter your ngrok authentication token: ").strip()
if not ngrok_token:
    raise ValueError("An ngrok authentication token is required")
ngrok.set_auth_token(ngrok_token)
del ngrok_token
print("ngrok authentication configured")

## 3. Optional: launch the WebUI

Run this cell for manual use. Skip it when you only need the GitHub Issues queue.

In [ ]:
import time
from urllib.error import URLError
from urllib.request import urlopen

WEBUI_PORT = 8501
WEBUI_LOG_PATH = Path("/content/moneyprinterturbo-webui.log")

previous_streamlit_proc = globals().get("streamlit_proc")
if previous_streamlit_proc is not None and previous_streamlit_proc.poll() is None:
    previous_streamlit_proc.terminate()
    previous_streamlit_proc.wait(timeout=10)
previous_streamlit_log = globals().get("streamlit_log")
if previous_streamlit_log is not None and not previous_streamlit_log.closed:
    previous_streamlit_log.close()
previous_webui_tunnel = globals().get("webui_tunnel")
if previous_webui_tunnel is not None:
    try:
        ngrok.disconnect(previous_webui_tunnel.public_url)
    except Exception:
        pass

streamlit_log = WEBUI_LOG_PATH.open("w", encoding="utf-8")
streamlit_proc = subprocess.Popen(
    [
        "uv", "run", "streamlit", "run", "webui/Main.py",
        f"--server.port={WEBUI_PORT}",
        "--server.address=0.0.0.0",
        "--browser.gatherUsageStats=False",
        "--client.toolbarMode=minimal",
        "--server.showEmailPrompt=False",
    ],
    cwd=REPO_DIR,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    text=True,
)

deadline = time.time() + 90
server_ready = False
while time.time() < deadline:
    if streamlit_proc.poll() is not None:
        break
    try:
        with urlopen(f"http://127.0.0.1:{WEBUI_PORT}/_stcore/health", timeout=2) as response:
            server_ready = response.status == 200
    except (URLError, TimeoutError):
        pass
    if server_ready:
        break
    time.sleep(2)

if not server_ready:
    streamlit_log.flush()
    recent_log = WEBUI_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(f"Streamlit failed to start. Recent log:\n{recent_log}")

webui_tunnel = ngrok.connect(addr=f"http://127.0.0.1:{WEBUI_PORT}", proto="http", bind_tls=True)
print("MoneyPrinterTurbo WebUI:")
print(webui_tunnel.public_url)
print(f"Server log: {WEBUI_LOG_PATH}")

## 4. Launch the GitHub Issues queue

Create a fine-grained GitHub token for `Equilibriumpress/MoneyPrinterTurbo` with **Metadata: read** and **Issues: read and write**. The worker accepts issues created by the token owner, creates its status labels, starts the API, and publishes temporary result links through ngrok.

In [ ]:
from getpass import getpass

API_PORT = 8080
API_LOG_PATH = Path("/content/moneyprinterturbo-api.log")
WORKER_LOG_PATH = Path("/content/moneyprinterturbo-github-worker.log")

github_token = getpass("Enter the fine-grained GitHub token: ").strip()
if not github_token:
    raise ValueError("A GitHub token is required")

for process_name in ("github_worker_proc", "api_proc"):
    previous_process = globals().get(process_name)
    if previous_process is not None and previous_process.poll() is None:
        previous_process.terminate()
        previous_process.wait(timeout=10)
for log_name in ("github_worker_log", "api_log"):
    previous_log = globals().get(log_name)
    if previous_log is not None and not previous_log.closed:
        previous_log.close()
previous_api_tunnel = globals().get("api_tunnel")
if previous_api_tunnel is not None:
    try:
        ngrok.disconnect(previous_api_tunnel.public_url)
    except Exception:
        pass

api_log = API_LOG_PATH.open("w", encoding="utf-8")
api_proc = subprocess.Popen(
    ["uv", "run", "python", "main.py"],
    cwd=REPO_DIR,
    stdout=api_log,
    stderr=subprocess.STDOUT,
    text=True,
)

deadline = time.time() + 120
api_ready = False
while time.time() < deadline:
    if api_proc.poll() is not None:
        break
    try:
        with urlopen(f"http://127.0.0.1:{API_PORT}/openapi.json", timeout=2) as response:
            api_ready = response.status == 200
    except (URLError, TimeoutError):
        pass
    if api_ready:
        break
    time.sleep(2)

if not api_ready:
    api_log.flush()
    recent_log = API_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(f"MoneyPrinterTurbo API failed to start. Recent log:\n{recent_log}")

api_tunnel = ngrok.connect(addr=f"http://127.0.0.1:{API_PORT}", proto="http", bind_tls=True)
worker_env = os.environ.copy()
worker_env["MPT_GITHUB_TOKEN"] = github_token
worker_env["MPT_GITHUB_REPOSITORY"] = "Equilibriumpress/MoneyPrinterTurbo"
worker_env["MPT_API_BASE"] = f"http://127.0.0.1:{API_PORT}/api/v1"
worker_env["MPT_PUBLIC_BASE"] = api_tunnel.public_url
worker_env["MPT_MAX_VIDEO_COUNT"] = "1"
del github_token

github_worker_log = WORKER_LOG_PATH.open("w", encoding="utf-8")
github_worker_proc = subprocess.Popen(
    ["uv", "run", "python", "scripts/github_issue_worker.py", "--poll-seconds", "20"],
    cwd=REPO_DIR,
    env=worker_env,
    stdout=github_worker_log,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(3)
if github_worker_proc.poll() is not None:
    github_worker_log.flush()
    recent_log = WORKER_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(f"GitHub worker failed to start. Recent log:\n{recent_log}")

print("GitHub Issues queue is running.")
print("Create a job: https://github.com/Equilibriumpress/MoneyPrinterTurbo/issues/new/choose")
print(f"Temporary result base URL: {api_tunnel.public_url}")
print(f"API docs: {api_tunnel.public_url}/docs")
print(f"Worker log: {WORKER_LOG_PATH}")
print(f"API log: {API_LOG_PATH}")

## 5. Check queue logs

Run this cell when a job does not move from queued to processing.

In [ ]:
print("Worker log:\n")
print(WORKER_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-6000:])
print("\nAPI log:\n")
print(API_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-6000:])